# 09 - Reshape into player A versus player B
This notebook reshapes the data into player A vs player B (rather than 'winner' and 'loser' columns).
Needs 05_match_dates to have been run (it does not technically need the weather notebooks).
It produces the table table tennis_clean.matches_ab, and a manifest recording its contents


## Install the pinned libraries

In [ ]:

# Read from a git tag rather than from main, so a later re-run installs exactly the same versions.
CODE_VERSION = "code-v1"      # the git tag of the code being run: create it on GitHub before running
REPO = "katiedevane/tennis-project"
REQUIREMENTS_URL = f"https://raw.githubusercontent.com/{REPO}/{CODE_VERSION}/requirements.txt"
!pip install -q -r $REQUIREMENTS_URL

# Shared provenance helpers (code version, environment, index of data versions), from the same tag
PROVENANCE_URL = f"https://raw.githubusercontent.com/{REPO}/{CODE_VERSION}/provenance.py"
!wget -q -O provenance.py $PROVENANCE_URL

## Sign in and load settings from Colab Secrets
Enable Notebook access for both secrets in this notebook.

In [ ]:
import os
from google.colab import auth, userdata

auth.authenticate_user()

PROJECT_ID = userdata.get("TENNIS_PROJECT_ID")
BUCKET_NAME = userdata.get("TENNIS_BUCKET_NAME")
REGION = "EU"        # the bucket and the BigQuery datasets must share this location
DATASETS = ["tennis_raw", "tennis_clean", "tennis_features"]
os.environ["CLOUDSDK_CORE_PROJECT"] = PROJECT_ID

assert PROJECT_ID and BUCKET_NAME, "A secret is empty: check its name and Notebook access"
print("Settings loaded from Colab Secrets.")

## Record the code version and set the random seed
The git tag, commit, Python environment and random seed of this run are added to every manifest this notebook writes (see provenance.py).
`SEED` is the one project-wide seed: `start_run` applies it, and it should be passed wherever a function takes a seed (e.g. `random_state=SEED`).

In [ ]:
from provenance import start_run, register, table_version, sha256, utc_stamp, SEED

NOTEBOOK = "09_matches_player_ab"
RUN_INFO = start_run(f"https://github.com/{REPO}", CODE_VERSION, NOTEBOOK)

## Helpers

In [ ]:
import pandas as pd
from google.cloud import bigquery, storage

bq = bigquery.Client(project=PROJECT_ID, location=REGION)
DATED = f"{PROJECT_ID}.tennis_clean.matches_dated"
AB = f"{PROJECT_ID}.tennis_clean.matches_ab"

def run(sql):
    config = bigquery.QueryJobConfig(maximum_bytes_billed=1_000_000_000)   # safety cap: 1 GB per query
    return bq.query(sql, job_config=config).to_dataframe()

pd.set_option("display.max_rows", 200, "display.width", 200)

## Build the table
 Each match is reshaped into **player A versus player B**, with the target `a_won` (1 if A won, else 0). This avoids building the label into the layout.

A is decided by a hash of the match ID. A rerun will give the same table.

**Columns known only after the match**  are flagged with `post_` and can only be used as history for later matches

In [ ]:
SALT = "ab-v1"     # part of the assignment rule; changing it redraws A/B

CONTEXT = ["match_id", "tourney_id", "tourney_name", "name_key", "tourney_date", "season",
           "tourney_level", "surface", "draw_size", "best_of", "round", "event_format",
           "match_date_min", "match_date_max"]
PLAYER_FIELDS = ["id", "name", "seed", "entry", "hand", "ht", "ioc", "age", "rank", "rank_points"]
STAT_FIELDS = ["ace", "df", "svpt", "1stIn", "1stWon", "2ndWon", "SvGms", "bpSaved", "bpFaced"]

parts = []
for f in PLAYER_FIELDS:                                   # known before the match
    parts.append(f"IF(a_is_winner, winner_{f}, loser_{f}) AS a_{f}")
    parts.append(f"IF(a_is_winner, loser_{f}, winner_{f}) AS b_{f}")
for f in STAT_FIELDS:                                     # only known after the match
    parts.append(f"IF(a_is_winner, w_{f}, l_{f}) AS post_a_{f}")
    parts.append(f"IF(a_is_winner, l_{f}, w_{f}) AS post_b_{f}")

sql = f"""
CREATE OR REPLACE TABLE `{AB}`
PARTITION BY DATE_TRUNC(tourney_date, MONTH)
CLUSTER BY tourney_id
OPTIONS (description = 'One row per match: player A vs player B (A chosen by hash of match_id), label a_won. Columns starting post_ are only known after the match.')
AS
WITH base AS (
  SELECT *, MOD(FARM_FINGERPRINT(CONCAT('{SALT}|', match_id)), 2) = 0 AS a_is_winner
  FROM `{DATED}`
)
SELECT {", ".join(CONTEXT)},
       {", ".join(parts)},
       IF(a_is_winner, 1, 0) AS a_won,
       score AS post_score,
       minutes AS post_minutes,
       is_walkover AS post_is_walkover,
       is_retirement AS post_is_retirement,
       is_default AS post_is_default
FROM base
"""
job = bq.query(sql); job.result()
print("Built", AB, "| bytes processed:", f"{job.total_bytes_processed:,}")

## Check that the new table matches its source


In [ ]:
cols = [f.name for f in bq.get_table(AB).schema]
print("columns:", len(cols))
print("winner/loser columns left:", [c for c in cols if "winner" in c or "loser" in c])     # expect []
print("post_ columns:", [c for c in cols if c.startswith("post_")])

print(run(f"""SELECT (SELECT COUNT(*) FROM `{AB}`) AS ab_rows,
                     (SELECT COUNT(*) FROM `{DATED}`) AS dated_rows,
                     (SELECT COUNT(DISTINCT match_id) FROM `{AB}`) AS unique_match_ids"""))

checks = [f"(a.a_{f} IS NOT DISTINCT FROM IF(a.a_won = 1, m.winner_{f}, m.loser_{f}) "
          f"AND a.b_{f} IS NOT DISTINCT FROM IF(a.a_won = 1, m.loser_{f}, m.winner_{f}))" for f in PLAYER_FIELDS]
print(run(f"""
SELECT COUNTIF(NOT ({" AND ".join(checks)})) AS inconsistent_rows,
       COUNTIF(a.a_id = a.b_id) AS same_player_both_sides
FROM `{AB}` a JOIN `{DATED}` m USING (match_id)
"""))

## Check A/B assignment is balanced and independent of result
- The share of matches won by A should be approx. 0.5 overall
- Player A should be the better-ranked player approx. as often as player B is.
- The better-ranked player should win > half the time

In [ ]:
print(run(f"SELECT COUNT(*) AS n, ROUND(AVG(a_won), 4) AS share_a_won FROM `{AB}`"))
print(run(f"SELECT season, COUNT(*) AS n, ROUND(AVG(a_won), 3) AS share_a_won FROM `{AB}` GROUP BY 1 ORDER BY 1"))
print(run(f"SELECT tourney_level, surface, COUNT(*) AS n, ROUND(AVG(a_won), 3) AS share_a_won FROM `{AB}` GROUP BY 1, 2 ORDER BY 1, 2"))

print(run(f"""
SELECT COUNTIF(a_rank < b_rank) AS a_better_ranked,
       COUNTIF(a_rank > b_rank) AS b_better_ranked,
       ROUND(AVG(IF(a_rank < b_rank, a_won, NULL)), 3) AS better_ranked_wins_when_A,
       ROUND(AVG(IF(a_rank > b_rank, 1 - a_won, NULL)), 3) AS better_ranked_wins_when_B
FROM `{AB}` WHERE a_rank IS NOT NULL AND b_rank IS NOT NULL
"""))

## Save a manifest
The table is no longer exported as a Parquet snapshot (that was a second copy of a BigQuery table). The manifest stores an ID of the table's contents under a version, so any later change is detected.

The Parquet files from earlier runs (`processed/matches_ab/v2/`) are no longer used and can be deleted.

In [ ]:
import json
from datetime import datetime, timezone

bucket = storage.Client(project=PROJECT_ID).bucket(BUCKET_NAME)
VERSION = "v2"
MANIFEST_PREFIX = f"manifests/matches_ab_{VERSION}_"

rows = int(run(f"SELECT COUNT(*) AS n FROM `{AB}`")["n"][0])
fingerprint = run(f"SELECT CAST(BIT_XOR(FARM_FINGERPRINT(TO_JSON_STRING(t))) AS STRING) AS fp FROM `{AB}` t")["fp"][0]
manifests = sorted(b.name for b in bucket.list_blobs(prefix=MANIFEST_PREFIX))

if manifests:
    manifest_path = manifests[0]
    recorded = json.loads(bucket.blob(manifest_path).download_as_text()).get("table_fingerprint")
    if recorded != fingerprint:
        raise RuntimeError(f"The table has changed since {VERSION} was recorded. Set VERSION to a new value and re-run.")
    print(f"{VERSION} already recorded and matches the table. Nothing to do.")
else:
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    manifest_path = f"{MANIFEST_PREFIX}{stamp}.json"
    bucket.blob(manifest_path).upload_from_string(
        json.dumps({"table": "tennis_clean.matches_ab", "version": VERSION, "rows": rows, "salt": SALT,
                    "table_fingerprint": fingerprint, **RUN_INFO}, indent=1),
        content_type="application/json")
    print("Manifest written:", manifest_path, "| rows:", rows)

register(bucket, "tennis_clean.matches_ab", VERSION,
         outputs=[{"table": "tennis_clean.matches_ab", "rows": rows, "fingerprint": fingerprint}],
         parents=[table_version(bq, DATED)], manifest_path=manifest_path)